In [ ]:
%pip uninstall torch torchvision torchaudio -y
%pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!rsync -a --info=progress2 /kaggle/input/notebooks/mustafamuhaimin/3mogenpretrain/ /kaggle/working/
!rsync -a /kaggle/usr/lib/notebooks/arifulislamshadhin/utilsgen/utilsgen.py /kaggle/working/utils.py



In [ ]:
pip install cattrs

In [ ]:
from pathlib import Path
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
import utils as utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(torch.__version__)
    print(torch.version.cuda)
    print(torch.cuda.get_device_name(0))
    print(torch.cuda.get_device_capability(0))

In [ ]:


from utils import Config
from utils import create_dataloader
from utils import EMAModel, _find_latest_checkpoint
from utils import FinetuneTrainer
from utils import FlowMatchingTrainer
from utils import (
    FeatureNormalizer,
    Features,
    x271_to_x72,
    x72_to_positions,
    x72_to_x271,
    x271_to_x68,
    x68_to_positions,
)


In [ ]:
config = Config()
config.device = device

# Paths
config.dataset_path    = Path("./dataset/humanml3d-subset")
config.checkpoint_dir  = Path("./checkpoints/phase3")
config.output_path     = Path("./output/phase3")

# Training
config.batch_size           = 128
config.effective_batch_size = 256
config._num_epochs          = 1000
config.curriculum           = None
config.horizon              = 80

# History window (frames preceding the target window — used as track_features context)
config.history_length = 40

# Flow matching
config.t_sampling_power = 0.0
config.cfg_dropout      = 0.1

config.val_interval = 25
config.checkpoint_interval = 100

# W&B
wandb_decoder_project   = "phase3_decoder"
wandb_predictor_project = "phase3_predictor"

# Track B Retraining Enhancements
config.use_prefix_history  = True    # Prefix in self-attention with RoPE + pure text cross-attention
config.loss_vel_weight     = 0.1     # Relative kinematic derivative loss (L_vel)

print(f"Config ready.  epochs={config.get_num_epochs()}  horizon={config.horizon}  history_length={config.history_length}")

In [ ]:
train_loader, normalizer = create_dataloader(config, "train", shuffle=True)
val_loader,   _          = create_dataloader(config, "val",   shuffle=False)
# --- FIX: Set horizon on both datasets ---
train_loader.dataset.set_horizon(config.horizon)
val_loader.dataset.set_horizon(config.horizon)


# Verify the batch has the expected keys
sample = next(iter(train_loader))
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"motion shape  : {sample['motion'].shape}         (B, T_target, 271)")
print(f"history shape : {sample['history_motion'].shape} (B, T_history=history_length, 271)")
assert "history_motion" in sample, "dataset.py did not emit history_motion — rebuild dataset"
del sample

In [ ]:
gc.collect()
torch.cuda.empty_cache()

pretrained_checkpoint = _find_latest_checkpoint(
    Path("./checkpoints/pretrain"), prefix="pretrain_best_val"
)
print(f"Pretrained checkpoint: {pretrained_checkpoint}")

# ── Decoder trainer ───────────────────────────────────────────────────────────
decoder_config = Config()
decoder_config.device              = device
decoder_config.dataset_path        = config.dataset_path
decoder_config.checkpoint_dir      = config.checkpoint_dir / "decoder"
decoder_config.output_path         = config.output_path    / "decoder"
decoder_config.batch_size          = config.batch_size
decoder_config.effective_batch_size= config.effective_batch_size
decoder_config._num_epochs         = config._num_epochs
decoder_config.curriculum          = config.curriculum
decoder_config.horizon             = config.horizon
decoder_config.history_length      = config.history_length
decoder_config.checkpoint_interval = config.checkpoint_interval

decoder_trainer = FinetuneTrainer(
    config=decoder_config,
    pretrained_checkpoint_path=pretrained_checkpoint,
    wandb_project=wandb_decoder_project,
)

# ── Predictor trainer ─────────────────────────────────────────────────────────
predictor_config = Config()
predictor_config.device              = device
predictor_config.dataset_path        = config.dataset_path
predictor_config.checkpoint_dir      = config.checkpoint_dir / "predictor"
predictor_config.output_path         = config.output_path    / "predictor"
predictor_config.batch_size          = config.batch_size
predictor_config.effective_batch_size= config.effective_batch_size
predictor_config._num_epochs         = config._num_epochs
predictor_config.curriculum          = config.curriculum
predictor_config.horizon             = config.horizon
predictor_config.history_length      = config.history_length
predictor_config.t_sampling_power    = config.t_sampling_power
predictor_config.cfg_dropout         = config.cfg_dropout
predictor_config.checkpoint_interval = config.checkpoint_interval
predictor_config.use_prefix_history  = True
predictor_config.loss_vel_weight     = 0.1

predictor_trainer = FlowMatchingTrainer(
    config=predictor_config,
    pretrained_checkpoint_path=pretrained_checkpoint,
    wandb_project=wandb_predictor_project,
)

print("Both trainers initialised.")

In [ ]:
num_epochs       = config.get_num_epochs()
acc_steps        = max(config.effective_batch_size // config.batch_size, 1)
gradient_clip    = float(config.gradient_clip)
use_amp          = (device.type == "cuda")
amp_dtype        = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)
t_power          = float(config.t_sampling_power)
cfg_dropout_prob = float(config.cfg_dropout)
ckpt_interval    = config.checkpoint_interval

dec_scaler  = torch.amp.GradScaler(device.type, enabled=use_amp)
pred_scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

steps_per_epoch = max(len(train_loader) // acc_steps, 1)
total_steps     = max(num_epochs * steps_per_epoch, 1)

dec_optimizer  = decoder_trainer.optimizer
pred_optimizer = predictor_trainer.optimizer
dec_scheduler  = decoder_trainer.lr_scheduler
pred_scheduler = predictor_trainer.lr_scheduler

dec_ckpt_dir  = decoder_config.checkpoint_dir
pred_ckpt_dir = predictor_config.checkpoint_dir
dec_ckpt_dir.mkdir(parents=True, exist_ok=True)
pred_ckpt_dir.mkdir(parents=True, exist_ok=True)


def compute_decoder_loss(
    decoded: torch.Tensor,
    motion: torch.Tensor,
    joints: torch.Tensor,
) -> torch.Tensor:
    """Reconstruction loss for 72D LatentDecoder with direct RIC position prediction."""
    global normalizer

    # Backwards compatibility with older checkpoints.
    if not hasattr(normalizer, "denormalize_x72"):
        normalizer = FeatureNormalizer(normalizer.mean, normalizer.std)

    prev_positions = joints[:, :-1, :].flatten(0, 1)
    prev_frames    = motion[:, :-1, :].flatten(0, 1)
    y_frames       = motion[:, 1:, :].flatten(0, 1)

    decoded_flat   = decoded.flatten(0, 1)
    decoded_denorm = normalizer.denormalize_x72(decoded_flat)

    y_72d = x271_to_x72(
        y_frames,
        normalizer,
        prev_positions=prev_positions,
        prev_x271=prev_frames,
    )
    y_72d_denorm = normalizer.denormalize_x72(y_72d)

    loss_root_y = F.mse_loss(
        decoded_denorm[:, :1],
        y_72d_denorm[:, :1],
    )

    loss_root_xz = F.mse_loss(
        decoded_denorm[:, 1:3],
        y_72d_denorm[:, 1:3],
    )

    loss_yaw = F.mse_loss(
        decoded_denorm[:, 3:5],
        y_72d_denorm[:, 3:5],
    )

    loss_ric = F.mse_loss(
        decoded_denorm[:, 5:68],
        y_72d_denorm[:, 5:68],
    )

    loss_contact = F.binary_cross_entropy_with_logits(
        decoded_flat[:, 68:72],
        y_frames[:, Features.CONTACTS],
    )

    dec_positions = x72_to_positions(
        decoded_flat,
        normalizer,
        prev_x271=prev_frames,
        prev_positions=prev_positions,
    )

    loss_joint = F.mse_loss(
        dec_positions,
        joints[:, 1:, :].flatten(0, 1),
    )

    return (
        1.0 * loss_root_y
        + 1.0 * loss_root_xz
        + 1.0 * loss_yaw
        + 1.0 * loss_ric
        + 0.5 * loss_contact
        + 1.0 * loss_joint
    )


print(
    f"Accumulation steps: {acc_steps}  |  "
    f"Steps/epoch: {steps_per_epoch}  |  "
    f"Total steps: {total_steps}"
)

In [ ]:
train_decoder   = False
train_predictor = True

# Only save checkpoints to disk during the final 100 epochs to conserve disk space
save_checkpoint_start_epoch = max(1, num_epochs - 450)

best_dec_val_loss  = float("inf")
best_pred_val_loss = float("inf")

target_encoder = decoder_trainer.ema_encoder.model
target_encoder.eval()

for epoch in range(1, num_epochs + 1):

    decoder_trainer.decoder.train()
    predictor_trainer.predictor.train()
    predictor_trainer.null_text_embedding.train()

    epoch_dec_loss  = 0.0
    epoch_pred_loss = 0.0
    num_batches_dec = 0
    num_batches_pred = 0

    dec_optimizer.zero_grad(set_to_none=True)
    pred_optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        motion_raw   = batch["motion"].to(device)          # (B, T_target, 271) raw
        history_raw  = batch["history_motion"].to(device)  # (B, T_history, 271) raw
        text_raw     = batch["text_clip"].to(device)
        joints       = batch["joints"].to(device)

        # Normalise both windows
        target_motion  = normalizer.normalize(motion_raw)   # (B, T_target, 271)
        history_motion = normalizer.normalize(history_raw)  # (B, T_history, 271)
        text_emb = text_raw[:, -1, :] if text_raw.ndim == 3 else text_raw  # (B, 512)
        B = target_motion.shape[0]

        # =============================================================================
        # SHARED: two separate frozen encoder passes — one per window, zeroed text
        # =============================================================================
        with torch.no_grad():
            zeros_target = torch.zeros_like(text_emb)  # zeroed text for target pass
            zeros_hist   = torch.zeros(B, text_emb.shape[1],
                                       device=device, dtype=target_motion.dtype)

            # z1: target latents (skip token-0, last encoder layer)
            raw_target = target_encoder(
                target_motion, zeros_target, mask=None, return_layer_outputs=True
            )  # (B, T_target, L, H_enc)
            z1 = raw_target[:, 1:, -1, :].detach()  # (B, T_target-1, H_enc)

            # track_features: history context (disjoint input — no leakage)
            track_features = target_encoder(
                history_motion, zeros_hist, mask=None, return_layer_outputs=False
            ).detach()  # (B, T_history, H_enc)

        # =============================================================================
        # STEP 1 — DECODER  (only if toggle button is set to ON)
        # =============================================================================
        if train_decoder:
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                decoded  = decoder_trainer.decoder(z1)                # (B, T_target-1, 72)
                dec_loss = compute_decoder_loss(decoded, target_motion, joints)

            dec_scaler.scale(dec_loss / acc_steps).backward()

            if (step + 1) % acc_steps == 0:
                dec_scaler.unscale_(dec_optimizer)
                torch.nn.utils.clip_grad_norm_(
                    decoder_trainer.decoder.parameters(), max_norm=gradient_clip
                )
                dec_scaler.step(dec_optimizer)
                dec_scaler.update()
                dec_optimizer.zero_grad(set_to_none=True)
                decoder_trainer.ema_decoder.update(decoder_trainer.decoder)
                dec_scheduler.step()

            epoch_dec_loss += dec_loss.item()
            num_batches_dec += 1

        # =============================================================================
        # STEP 2 — PREDICTOR  (only if toggle button is set to ON)
        # =============================================================================
        if train_predictor:
            text_clip = batch.get("text_clip", None)
            if "captions" in batch:
                text_clip = predictor_trainer.clip_encoder.encode_sequence(batch["captions"])
            elif text_clip is not None:
                text_clip = text_clip.to(device)

            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                # Consolidated rectified flow training step using precomputed shared encoder outputs
                pred_loss, _t_pred = predictor_trainer.compute_flow_loss(
                    target_motion=target_motion,
                    history_motion=history_motion,
                    text_emb=text_clip,
                    is_training=True,
                    z1=z1,
                    track_features=track_features,
                    history_valid_length=batch.get("history_valid_length", None),
                )
            pred_scaler.scale(pred_loss / acc_steps).backward()

            if (step + 1) % acc_steps == 0:
                pred_scaler.unscale_(pred_optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(predictor_trainer.predictor.parameters()) +
                    list(predictor_trainer.null_text_embedding.parameters()),
                    max_norm=gradient_clip,
                )
                pred_scaler.step(pred_optimizer)
                pred_scaler.update()
                pred_optimizer.zero_grad(set_to_none=True)
                predictor_trainer.ema_predictor.update(predictor_trainer.predictor)
                pred_scheduler.step()

            epoch_pred_loss += pred_loss.item()
            num_batches_pred += 1

    avg_dec  = epoch_dec_loss  / max(num_batches_dec, 1) if num_batches_dec > 0 else 0.0
    avg_pred = epoch_pred_loss / max(num_batches_pred, 1) if num_batches_pred > 0 else 0.0

    dec_lr_str  = f"{dec_scheduler.get_last_lr()[0]:.2e}" if train_decoder else "PAUSED"
    pred_lr_str = f"{pred_scheduler.get_last_lr()[0]:.2e}" if train_predictor else "PAUSED"

    print(
        f"Epoch {epoch:4d}/{num_epochs} | "
        f"dec_loss={avg_dec:.4f} | pred_loss={avg_pred:.4f} | "
        f"dec_lr={dec_lr_str} | "
        f"pred_lr={pred_lr_str}"
    )

    # ── Validation ────────────────────────────────────────────────────────────
    if epoch % config.val_interval == 0:
        decoder_trainer.decoder.eval()
        decoder_trainer.ema_decoder.model.eval()
        predictor_trainer.ema_predictor.model.eval()

        val_dec_loss  = 0.0
        val_pred_loss = 0.0
        val_batches   = 0

        with torch.no_grad():
            for vbatch in val_loader:
                vmotion_raw  = vbatch["motion"].to(device)
                vhistory_raw = vbatch["history_motion"].to(device)
                vtext_raw    = vbatch["text_clip"].to(device)
                vjoints      = vbatch["joints"].to(device)

                vtarget  = normalizer.normalize(vmotion_raw)
                vhistory = normalizer.normalize(vhistory_raw)
                vB       = vtarget.shape[0]

                # Zeroed text for frozen encoder passes
                vz_text = torch.zeros(vB, 512, device=device, dtype=vtarget.dtype)
                vh_text = torch.zeros(vB, 512, device=device, dtype=vtarget.dtype)

                vraw_target = target_encoder(
                    vtarget, vz_text, mask=None, return_layer_outputs=True
                )
                vz1             = vraw_target[:, 1:, -1, :]
                vtrack_features = target_encoder(
                    vhistory, vh_text, mask=None, return_layer_outputs=False
                )

                # Consistent validation text encoding matching training step (77-token sequence)
                if "captions" in vbatch:
                    vtext_clip = predictor_trainer.clip_encoder.encode_sequence(vbatch["captions"])
                elif vtext_raw is not None:
                    vtext_clip = vtext_raw.to(device)
                else:
                    vtext_clip = None

                # Validate decoder regardless (showing current performance)
                with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                    vdecoded = decoder_trainer.ema_decoder.model(vz1)
                    vdec_l   = compute_decoder_loss(vdecoded, vtarget, vjoints)
                val_dec_loss += vdec_l.item()

                with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                    vpred_loss, _ = predictor_trainer.compute_flow_loss(
                        vtarget, vhistory, vtext_clip,
                        is_training=False,
                        z1=vz1,
                        track_features=vtrack_features,
                        history_valid_length=vbatch.get("history_valid_length", None),
                    )
                val_pred_loss += vpred_loss.item()

                val_batches += 1
                if val_batches >= config.val_batches:
                    break

        avg_vdec  = val_dec_loss  / max(val_batches, 1)
        avg_vpred = val_pred_loss / max(val_batches, 1)
        print(f"          Val | dec_val={avg_vdec:.4f} | pred_val={avg_vpred:.4f}")

        # Save best decoder checkpoint ONLY if training AND during final 100 epochs
        if avg_vdec < best_dec_val_loss:
            best_dec_val_loss = avg_vdec
            if train_decoder and epoch >= save_checkpoint_start_epoch:
                dec_ckpt = dec_ckpt_dir / f"phase3_decoder_best_val_{decoder_trainer.session_id}.pt"
                torch.save({
                    "encoder":       decoder_trainer.encoder.state_dict(),
                    "encoder_ema":   decoder_trainer.ema_encoder.state_dict(),
                    "decoder":       decoder_trainer.decoder.state_dict(),
                    "decoder_ema":   decoder_trainer.ema_decoder.state_dict(),
                    "epoch":         epoch,
                    "best_val_loss": best_dec_val_loss,
                    "config":        decoder_config.to_dict(),
                }, dec_ckpt)
                print(f"          ✓ New best decoder checkpoint: {dec_ckpt.name}")

        # Save best predictor checkpoint ONLY if training AND during final 100 epochs
        if avg_vpred < best_pred_val_loss:
            best_pred_val_loss = avg_vpred
            if train_predictor:  # Save best predictor whenever improved (safe against Kaggle timeouts)
                pred_ckpt = pred_ckpt_dir / f"phase3_predictor_best_val_{predictor_trainer.session_id}.pt"
                torch.save({
                    "encoder":             predictor_trainer.encoder.state_dict(),
                    "encoder_ema":         predictor_trainer.ema_encoder.state_dict(),
                    "predictor":           predictor_trainer.predictor.state_dict(),
                    "predictor_ema":       predictor_trainer.ema_predictor.state_dict(),
                    "null_text_embedding": predictor_trainer.null_text_embedding.state_dict(),
                    "epoch":               epoch,
                    "best_val_loss":       best_pred_val_loss,
                    "config":              predictor_config.to_dict(),
                }, pred_ckpt)
                print(f"          ✓ New best predictor checkpoint: {pred_ckpt.name}")

    if epoch % ckpt_interval == 0 and epoch >= save_checkpoint_start_epoch:
        if train_decoder:
            torch.save({
                "decoder":     decoder_trainer.decoder.state_dict(),
                "decoder_ema": decoder_trainer.ema_decoder.state_dict(),
                "epoch":       epoch,
            }, dec_ckpt_dir / f"phase3_decoder_latest_{decoder_trainer.session_id}.pt")

        if train_predictor:
            torch.save({
                "predictor":           predictor_trainer.predictor.state_dict(),
                "predictor_ema":       predictor_trainer.ema_predictor.state_dict(),
                "null_text_embedding": predictor_trainer.null_text_embedding.state_dict(),
                "epoch":               epoch,
            }, pred_ckpt_dir / f"phase3_predictor_latest_{predictor_trainer.session_id}.pt")

print("Phase 3 training complete.")
print(f"Best decoder   val loss: {best_dec_val_loss:.4f}")
print(f"Best predictor val loss: {best_pred_val_loss:.4f}")

In [ ]:
# =============================================================================
# DIAGNOSTIC 1: Latent Quality Check
# Run this in a notebook cell after your trainers are initialized.
# It compares predictor output z1_pred vs ground-truth z1_gt
# to quantify how well the predictor has learned.
# =============================================================================

import torch
import torch.nn.functional as F
import numpy as np

# ── Grab a validation batch ──────────────────────────────────────────────────
if 'val_iterator' not in dir():
    val_iterator = iter(val_loader)
try:
    val_batch = next(val_iterator)
except StopIteration:
    val_iterator = iter(val_loader)
    val_batch = next(val_iterator)

# ── Setup ────────────────────────────────────────────────────────────────────
device = predictor_trainer.device
encoder = predictor_trainer.ema_encoder.model
predictor = predictor_trainer.ema_predictor.model
encoder.eval()
predictor.eval()

# ── 1. Compute ground-truth z1 from encoder ──────────────────────────────────
motion_raw = val_batch["motion"].to(device)
history_raw = val_batch["history_motion"].to(device)
target_motion = normalizer.normalize(motion_raw)
history_motion = normalizer.normalize(history_raw)

with torch.no_grad():
    zeros_text = torch.zeros(target_motion.shape[0], 512, device=device, dtype=target_motion.dtype)
    
    # Ground truth latents
    raw_target = encoder(target_motion, zeros_text, mask=None, return_layer_outputs=True)
    z1_gt = raw_target[:, 1:, -1, :]  # (B, T-1, H_enc=512)
    
    # History track features  
    track_features = encoder(history_motion, zeros_text, mask=None, return_layer_outputs=False)
    
    # Normalize z1 the same way training does
    z1_gt_norm = predictor_trainer.normalize_latent(z1_gt)

B, T, H = z1_gt_norm.shape
print(f"Batch: B={B}, T_target={T}, H_enc={H}")
print(f"z1_gt_norm stats: mean={z1_gt_norm.mean().item():.4f}, std={z1_gt_norm.std().item():.4f}")
print(f"  per-dim std range: [{z1_gt_norm.std(dim=(0,1)).min().item():.4f}, {z1_gt_norm.std(dim=(0,1)).max().item():.4f}]")

# ── 2. Run predictor: full ODE integration (same as pipeline.py) ─────────────
with torch.no_grad():
    # Text embedding
    if "captions" in val_batch:
        text_seq = predictor_trainer.clip_encoder.encode_sequence(val_batch["captions"]).to(device)
    else:
        text_raw = val_batch["text_clip"].to(device)
        text_seq = text_raw[:, -1:, :] if text_raw.ndim == 3 else text_raw.unsqueeze(1)
    
    text_pooled = text_seq.mean(dim=1)  # (B, 512)
    
    # Track B: Pure text in cross-attention + prefix history in self-attention with RoPE
    use_prefix = getattr(predictor.config.predictor_config, "use_prefix_history", True)
    if use_prefix:
        cross_cond = text_seq
        hist_states = track_features
        key_padding_mask = None
    else:
        cross_cond = torch.cat([text_seq, track_features], dim=1)
        hist_states = None
        key_padding_mask = None

    # ODE integration from noise (Midpoint solver, uniform schedule power=1.0)
    z_t = torch.randn(B, T, H, device=device)
    num_steps = 20
    power = 1.0
    s = torch.linspace(0.0, 1.0, num_steps + 1, device=device)
    tau = 1.0 - (1.0 - s).pow(power)
    
    for step in range(num_steps):
        t_start = tau[step].expand(B)
        dt = tau[step + 1] - tau[step]
        
        v_pred, _, _ = predictor(
            noisy_states=z_t,
            timesteps=t_start,
            track_features=cross_cond,
            text_embedding=text_pooled,
            key_padding_mask=key_padding_mask,
            history_states=hist_states,
        )
        z_mid = z_t + v_pred * (0.5 * dt)
        t_mid = t_start + 0.5 * dt
        v_mid, _, _ = predictor(
            noisy_states=z_mid,
            timesteps=t_mid,
            track_features=cross_cond,
            text_embedding=text_pooled,
            key_padding_mask=key_padding_mask,
            history_states=hist_states,
        )
        z_t = z_t + v_mid * dt
    
    z1_pred_norm = z_t  # predicted z1 in normalized space

# ── 3. Compute metrics ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("DIAGNOSTIC 1: Latent Quality Check")
print("=" * 70)

# Overall MSE
mse = F.mse_loss(z1_pred_norm, z1_gt_norm).item()
print(f"\n1. Overall MSE(z1_pred, z1_gt) in normalized space: {mse:.6f}")

# Baseline: MSE of pure random noise vs z1_gt
z_random = torch.randn_like(z1_gt_norm)
mse_random = F.mse_loss(z_random, z1_gt_norm).item()
print(f"   Baseline MSE(random_noise, z1_gt):               {mse_random:.6f}")
print(f"   Ratio (pred/random): {mse/mse_random:.4f}  (< 0.5 = predictor learned something, > 0.8 = nearly random)")

# Mean of zeros baseline
mse_zeros = F.mse_loss(torch.zeros_like(z1_gt_norm), z1_gt_norm).item()
print(f"   Baseline MSE(zeros, z1_gt):                      {mse_zeros:.6f}")

# Per-frame cosine similarity
cos_sim = F.cosine_similarity(z1_pred_norm.flatten(0, 1), z1_gt_norm.flatten(0, 1), dim=-1)
print(f"\n2. Cosine similarity (frame-level):")
print(f"   Mean: {cos_sim.mean().item():.4f}")
print(f"   Std:  {cos_sim.std().item():.4f}")
print(f"   Min:  {cos_sim.min().item():.4f}")
print(f"   Max:  {cos_sim.max().item():.4f}")
print(f"   (Good: > 0.5, Bad: < 0.2, Random: ~0.0)")

# Per-channel correlation
z1_pred_flat = z1_pred_norm.flatten(0, 1).cpu().numpy()  # (B*T, H)
z1_gt_flat = z1_gt_norm.flatten(0, 1).cpu().numpy()
per_channel_corr = []
for i in range(H):
    corr = np.corrcoef(z1_pred_flat[:, i], z1_gt_flat[:, i])[0, 1]
    per_channel_corr.append(corr if not np.isnan(corr) else 0.0)
per_channel_corr = np.array(per_channel_corr)

print(f"\n3. Per-channel Pearson correlation:")
print(f"   Mean:   {np.mean(per_channel_corr):.4f}")
print(f"   Median: {np.median(per_channel_corr):.4f}")
print(f"   > 0.5:  {(per_channel_corr > 0.5).sum()}/{H} channels")
print(f"   > 0.3:  {(per_channel_corr > 0.3).sum()}/{H} channels")
print(f"   < 0.1:  {(per_channel_corr < 0.1).sum()}/{H} channels")
print(f"   (Good: most channels > 0.5, Bad: most < 0.1)")

# Distribution comparison
print(f"\n4. Distribution of predicted vs GT latents:")
print(f"   GT   mean: {z1_gt_norm.mean().item():.4f}, std: {z1_gt_norm.std().item():.4f}")
print(f"   Pred mean: {z1_pred_norm.mean().item():.4f}, std: {z1_pred_norm.std().item():.4f}")
print(f"   GT   per-dim mean range:  [{z1_gt_norm.mean(dim=(0,1)).min().item():.4f}, {z1_gt_norm.mean(dim=(0,1)).max().item():.4f}]")
print(f"   Pred per-dim mean range:  [{z1_pred_norm.mean(dim=(0,1)).min().item():.4f}, {z1_pred_norm.mean(dim=(0,1)).max().item():.4f}]")
print(f"   GT   per-dim std range:   [{z1_gt_norm.std(dim=(0,1)).min().item():.4f}, {z1_gt_norm.std(dim=(0,1)).max().item():.4f}]")
print(f"   Pred per-dim std range:   [{z1_pred_norm.std(dim=(0,1)).min().item():.4f}, {z1_pred_norm.std(dim=(0,1)).max().item():.4f}]")

# ── 4. Single-step denoising test ────────────────────────────────────────────
print("\n" + "=" * 70)
print("DIAGNOSTIC 2: Single-step denoising from near-clean latent")
print("=" * 70)

with torch.no_grad():
    for t_val in [0.95, 0.8, 0.5, 0.2]:
        z0 = torch.randn_like(z1_gt_norm)
        z_noisy = (1 - t_val) * z0 + t_val * z1_gt_norm
        t_tensor = torch.full((B,), t_val, device=device)
        
        v_pred, _, _ = predictor(
            noisy_states=z_noisy,
            timesteps=t_tensor,
            track_features=cross_cond,
            text_embedding=text_pooled,
            key_padding_mask=key_padding_mask,
            history_states=hist_states,
        )
        
        # With perfect v_pred, z_noisy + v_pred * (1 - t_val) should equal z1_gt_norm
        # Because v* = z1 - z0, and z_t = (1-t)*z0 + t*z1
        # So z1 = z_t + v*(1-t) only if we integrate from t to 1
        # Actually for single step: z1_pred = z_noisy + v_pred * (1 - t_val)
        z1_denoised = z_noisy + v_pred * (1.0 - t_val)
        
        mse_denoised = F.mse_loss(z1_denoised, z1_gt_norm).item()
        cos_denoised = F.cosine_similarity(z1_denoised.flatten(0,1), z1_gt_norm.flatten(0,1), dim=-1).mean().item()
        
        print(f"   t={t_val:.2f}: MSE={mse_denoised:.6f}, CosSim={cos_denoised:.4f}  (t=1 is clean, t=0 is pure noise)")

# ── 5. Velocity prediction quality ──────────────────────────────────────────
print("\n" + "=" * 70)
print("DIAGNOSTIC 3: Velocity prediction quality at different timesteps")
print("=" * 70)

with torch.no_grad():
    for t_val in [0.95, 0.8, 0.5, 0.2, 0.05]:
        z0 = torch.randn_like(z1_gt_norm)
        z_noisy = (1 - t_val) * z0 + t_val * z1_gt_norm
        v_true = z1_gt_norm - z0  # true velocity field
        
        t_tensor = torch.full((B,), t_val, device=device)
        v_pred, _, _ = predictor(
            noisy_states=z_noisy,
            timesteps=t_tensor,
            track_features=cross_cond,
            text_embedding=text_pooled,
            key_padding_mask=key_padding_mask,
            history_states=hist_states,
        )
        
        v_mse = F.mse_loss(v_pred, v_true).item()
        v_cos = F.cosine_similarity(v_pred.flatten(0,1), v_true.flatten(0,1), dim=-1).mean().item()
        
        # Also check: is v_pred just predicting the mean velocity?
        v_pred_norm = v_pred.flatten(0,1).norm(dim=-1).mean().item()
        v_true_norm = v_true.flatten(0,1).norm(dim=-1).mean().item()
        
        print(f"   t={t_val:.2f}: v_MSE={v_mse:.4f}, v_CosSim={v_cos:.4f}, |v_pred|={v_pred_norm:.4f}, |v_true|={v_true_norm:.4f}")

# ── 6. Check if predictor is mode-collapsing ────────────────────────────────
print("\n" + "=" * 70)
print("DIAGNOSTIC 4: Mode collapse check")
print("=" * 70)

with torch.no_grad():
    # Run 3 different random seeds and see if predictions differ
    preds = []
    for seed in range(3):
        torch.manual_seed(seed + 42)
        z_t_mc = torch.randn(B, T, H, device=device)
        for step in range(num_steps):
            t_start = tau[step].expand(B)
            dt = tau[step + 1] - tau[step]
            v_pred, _, _ = predictor(
                noisy_states=z_t_mc, timesteps=t_start,
                track_features=cross_cond, text_embedding=text_pooled,
                key_padding_mask=key_padding_mask, history_states=hist_states,
            )
            z_t_mc = z_t_mc + v_pred * dt
        preds.append(z_t_mc)
    
    # Compare pairs
    for i in range(3):
        for j in range(i+1, 3):
            diff = F.mse_loss(preds[i], preds[j]).item()
            cos = F.cosine_similarity(preds[i].flatten(0,1), preds[j].flatten(0,1), dim=-1).mean().item()
            print(f"   Seed {i} vs {j}: MSE={diff:.6f}, CosSim={cos:.4f}")
    
    print(f"   (If all preds are nearly identical → mode collapse)")
    print(f"   (If very different but all bad → predictor outputs noise)")

# ── 7. Training loss sanity ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("DIAGNOSTIC 5: Current training loss value")
print("=" * 70)

with torch.no_grad():
    # Compute flow loss on this batch
    if "captions" in val_batch:
        text_emb_loss = predictor_trainer.clip_encoder.encode_sequence(val_batch["captions"]).to(device)
    else:
        text_raw = val_batch["text_clip"].to(device)
        text_emb_loss = text_raw[:, -1:, :] if text_raw.ndim == 3 else text_raw.unsqueeze(1)
    
    hist_valid = val_batch.get("history_valid_length", None)
    loss, t_vec = predictor_trainer.compute_flow_loss(
        target_motion=target_motion,
        history_motion=history_motion,
        text_emb=text_emb_loss,
        is_training=False,
        history_valid_length=hist_valid,
    )
    print(f"   Current validation flow loss: {loss.item():.6f}")
    
    # What would a random predictor's loss be?
    # v_target = z1_norm - z0, which has roughly ||v|| ~ sqrt(2*H) for unit Gaussians
    # MSE of random v_pred vs v_target ≈ 2.0 (since both ~N(0,1) per dim)
    print(f"   Expected random predictor loss: ~2.0 (for normalized latent)")
    print(f"   Expected converged loss: < 0.5 (ideally < 0.2)")

print("\n" + "=" * 70)
print("DIAGNOSIS COMPLETE — Review numbers above to determine next steps")
print("=" * 70)